In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)
from jax_sn.operators.transport_operator import TransportOperatorBatched, TransportOperatorMultipleAnglesVmap, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorMultipleAngles
from jax_sn.domain.cross_sections import CrossSectionsIndexed
import equinox as eqx
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
import lineax as lx

In [3]:
from finite_elements.test_breeding_blanket import bb_data_dict, base_region_sizes, base_resolutions,  _get_bb_args, _setup_bb_args, _get_bb_args


def setup_mgop(dim, degree):
    MGOP, source_eg0, solution_domain = _setup_bb_args(dim, degree, base_region_sizes(), base_resolutions(), 5.0, 10, 3)
    transport_operator                = TransportOperatorMultipleAnglesVmap(TransportOperatorBatchedPallas(MGOP.transport_operator.solution_domain_data, MGOP.transport_operator.topological_sorted_angle, 0, 100))
    MGOP                              = eqx.tree_at(lambda t : t.transport_operator, MGOP, transport_operator)
    return MGOP, source_eg0, solution_domain

def setup_mgop_mod_theta(dim, degree, xs_data_indices, dx):
    MGOP, source_eg0, solution_domain = setup_mgop(dim, degree)    

    total_xs = MGOP.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    scattering_xs = MGOP.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data.scattering    
    if xs_data_indices[0] == 'total':
        total_xs = total_xs.at[xs_data_indices[1]].add(dx)
        
    elif xs_data_indices[0] == 'scattering':
        scattering_xs = scattering_xs.at[xs_data_indices[1]].add(dx)
    else:
        raise ValueError("Invalid xs_data_indices")    
    
    sgop = MGOP.to_single_group_operator(0)

    xs_data_new = CrossSectionData(total_xs, scattering_xs)
    sgop_new = eqx.tree_at(lambda sgop : [sgop.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data, sgop.inscatter_operator.cross_sections_indexed.cross_section_data], sgop,[xs_data_new,xs_data_new])
    return sgop_new, source_eg0, solution_domain


MGOP, source_eg0, solution_domain = setup_mgop(2,2)

MGOP = MGOP.convert_to_lower_block_triangular()

In [4]:
from jax_sn.operators.moment_operator import MomentOperator
from jax_sn.operators.transport_operator import TransportOperatorMultipleAngles
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.operators.scatter_operator.gather_scatter_operator_imp import GatherScatterOperator
from jax_sn.operators.moment_operator import DiscreteOperator
from jax_sn.operators.multi_group_operator import MultiGroupOperator
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver
from jax_sn.tools.memory_analysis import MultiGroupOperatorSigmaAllParametrised, _filter_jit_f_to_memory, OperatorAnalysis, pytree_nbytes


In [5]:
theta_base = MultiGroupOperatorSigmaAllParametrised.theta_base(MGOP)
MGOP_re = MultiGroupOperatorSigmaAllParametrised.reconstruct_operator(MGOP, theta_base)

rhs_base = jnp.ones_like(MGOP.in_structure())
bicgstab_solver = lx.BiCGStab(1e-12, 1e-12, max_steps = 160)


In [6]:
# We want to isolate the batching here. This is a memory cost independent of the operator size.
MGOP_no_batch= eqx.tree_at(lambda op : op.transport_operator.transport_operator.batch_size , MGOP, replace = 1)

In [67]:
@eqx.filter_jit
def solve_fwd(mgop, rhs, solver):
    return lx.linear_solve(mgop, rhs, solver=solver).value


@eqx.filter_jit
def solve_lower_tri(mgop, rhs, solver):
    ltri_solver=  MultiGroupLowerTriangularBlockSolver(solver)
    return lx.linear_solve(mgop, rhs, solver=ltri_solver).value

@eqx.filter_jit
def solve_bwd(mgop, rhs, solver):    
    return lx.linear_solve(mgop.transpose(), rhs, solver=solver).value

@eqx.filter_jit
def solve_bwd_lower_tri(mgop, rhs, solver):    
    ltri_solver=  MultiGroupLowerTriangularBlockSolver(solver)
    return lx.linear_solve(mgop.transpose(), rhs, solver=ltri_solver).value

def print_costs_solves(MGOP, rhs_base, bicgstab_solver):
    sgop = MGOP.to_single_group_operator(0)
    rhs_base = jnp.ones_like(MGOP.in_structure())
    bicgstab_solver = lx.BiCGStab(1e-12, 1e-12, max_steps = 160)
    print("Transport Operator")
    print(OperatorAnalysis.analyze_memory(sgop.transport_operator, "inv"))
    print("Single group operator")
    print(OperatorAnalysis.analyze_memory(sgop, "mv"))
    print("Multi group operator")
    print(OperatorAnalysis.analyze_memory(MGOP, "mv"))
    print("Forward solve")
    print(_filter_jit_f_to_memory(solve_fwd, MGOP, rhs_base, bicgstab_solver ))
    print("Forward tri solve")
    print(_filter_jit_f_to_memory(solve_lower_tri, MGOP, rhs_base, bicgstab_solver ))

    print("Backward solve")
    print(_filter_jit_f_to_memory(solve_bwd, MGOP, rhs_base, bicgstab_solver ))
    print("Backward tri solve")
    print(_filter_jit_f_to_memory(solve_bwd_lower_tri, MGOP, rhs_base, bicgstab_solver ))
print_costs_solves(MGOP_no_batch, rhs_base, bicgstab_solver)


Transport Operator
{'peak': 6.268035888671875, 'argument': 2.34576416015625, 'output': 1.6479721069335938, 'temp': 2.2764968872070312}
Single group operator
{'peak': 7.3433685302734375, 'argument': 1.575103759765625, 'output': 0.8239974975585938, 'temp': 5.495491027832031}
Multi group operator
{'peak': 12.306617736816406, 'argument': 4.047027587890625, 'output': 3.2959213256835938, 'temp': 4.967231750488281}
Forward solve
{'peak': 33.7638635635376, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 26.37939453125}
Forward tri solve
{'peak': 19.500877380371094, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 13.765388488769531}
Backward solve
{'peak': 38.984429359436035, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 32.150634765625}
Backward tri solve
{'peak': 24.74176025390625, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 18.71265411376953}


In [8]:


@eqx.filter_jit
def solve_result(theta, MGOP, rhs_base, ltri_solver):    

    MGOP_new = MultiGroupOperatorSigmaAllParametrised.reconstruct_operator(MGOP, theta)
    solution_value =  lx.linear_solve(MGOP_new, rhs_base, solver=ltri_solver).value
    return jnp.sum(solution_value)

grad_theta = eqx.filter_jit(jax.grad(solve_result, argnums=0))

ltri_solver = MultiGroupLowerTriangularBlockSolver(bicgstab_solver)

In [75]:
from jax_sn.tools.memory_analysis import OperatorThetaParametrised
from jax_sn.solution_domain import PrecomputedGeometricData
from jax_sn.domain.cross_sections import CrossSectionData
from jax_sn.domain.geometric_domain import compute_face_normal_vectors_from_3d_simplices
from jax_sn.solution_domain.basix_fem import _precompute_jacobians_simplex

class EverythingTheta(eqx.Module):
    vertices : jnp.ndarray 
    cross_section_data: CrossSectionData

class MultiGroupOperatorEverythingParametrised(OperatorThetaParametrised):
    '''
    Parametrisation case for total operators parameterised by the total and scattering cross section.
    '''
    @staticmethod
    def theta_base(operator ):
        return EverythingTheta(operator.transport_operator.solution_domain_data.domain.vertices, operator.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data)
    
    @staticmethod
    def reconstruct_operator(operator, theta):

        # We have to replace:
        # 1. cross sections
        xs_idxed             = eqx.tree_at(lambda xs_idx : xs_idx.cross_section_data, operator.transport_operator.solution_domain_data.domain.cross_sections_indexed, replace = theta.cross_section_data)                
        operator_xs_replaced =  eqx.tree_at(operator.cross_sections_indexed_loc, operator, replace_fn= lambda _ :  xs_idxed)

        # 2. domain itself
        domain = operator_xs_replaced.transport_operator.solution_domain_data.domain
        #       -  Face normals in the domain & vertices in the domain (although they don't do anything anymore i think...)
        face_normals = compute_face_normal_vectors_from_3d_simplices(theta.vertices[domain.connectivity], domain.face_template)
        domain_replaced = eqx.tree_at(lambda d : [d.vertices, d.face_normals], domain, replace = [theta.vertices, face_normals])        
        operator_xs_dom_replaced = eqx.tree_at(lambda op : op.transport_operator.transport_operator.solution_domain_data.domain, operator_xs_replaced, replace = domain_replaced)

        # 3. precomputed geometric data
        precomputed_geometric_data_comp = _precompute_jacobians_simplex(domain_replaced)
        
        return eqx.tree_at(operator_xs_dom_replaced.precomputed_geometric_data_loc, operator_xs_dom_replaced, replace_fn= lambda _ : precomputed_geometric_data_comp)


theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)




@eqx.filter_jit
def solve_result_everything(theta, MGOP, rhs_base, ltri_solver):    
    MGOP_new = MultiGroupOperatorEverythingParametrised.reconstruct_operator(MGOP, theta)
    solution_value =  lx.linear_solve(MGOP_new, rhs_base, solver=ltri_solver).value
    return jnp.sum(solution_value)

grad_theta_everything = eqx.filter_jit(jax.grad(solve_result_everything, argnums=0))

In [76]:
print("Everything parametrised")
theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)
print("Tri solver (batch 100)", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP, rhs_base, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP_no_batch, rhs_base, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP, rhs_base, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(solve_result_everything,theta_everything_base, MGOP_no_batch, rhs_base, bicgstab_solver))


print("\nGradients:")
print("Tri solver (batch 100)", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP, rhs_base, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP_no_batch, rhs_base, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP, rhs_base, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP_no_batch, rhs_base, bicgstab_solver))


Everything parametrised
Tri solver (batch 100) {'peak': 36.10141086578369, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 32.15160369873047}
Tri solver (batch 1) {'peak': 19.42992401123047, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 17.12718963623047}
Total solver (batch 100 ) {'peak': 49.35383701324463, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 47.31932067871094}
Total solver (batch 1) {'peak': 33.692917823791504, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 29.741439819335938}

Gradients:
Tri solver (batch 100) {'peak': 151.82408142089844, 'argument': 3.9641876220703125, 'output': 0.01050567626953125, 'temp': 148.05361938476562}
Tri solver (batch 1) {'peak': 143.65656280517578, 'argument': 3.9641876220703125, 'output': 0.01050567626953125, 'temp': 139.71890258789062}
Total solver (batch 100 ) {'peak': 159.49404907226562, 'argument': 3.9641876220703125, 'output': 0.01050567626953125, 'temp':

In [63]:
print("Sigma parametrised")
print("Tri solver (batch 100)", _filter_jit_f_to_memory(solve_result, theta_base, MGOP, rhs_base, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(solve_result, theta_base, MGOP_no_batch, rhs_base, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(solve_result, theta_base, MGOP, rhs_base, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(solve_result, theta_base, MGOP_no_batch, rhs_base, bicgstab_solver))

print("\nGradients:")
print("Tri solver (batch 100)", _filter_jit_f_to_memory(grad_theta, theta_base, MGOP, rhs_base, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(grad_theta, theta_base, MGOP_no_batch, rhs_base, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(grad_theta, theta_base, MGOP, rhs_base, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(grad_theta, theta_base, MGOP_no_batch, rhs_base, bicgstab_solver))

Sigma parametrised


AttributeError: 'EverythingTheta' object has no attribute 'assert_n_energy_groups'

In [73]:
print("Everything parametrised")
theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)
print("Tri solver (batch 100)", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP, rhs_base, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP_no_batch, rhs_base, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP, rhs_base, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(solve_result_everything,theta_everything_base, MGOP_no_batch, rhs_base, bicgstab_solver))


print("\nGradients:")
print("Tri solver (batch 100)", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP, rhs_base, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP_no_batch, rhs_base, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP, rhs_base, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP_no_batch, rhs_base, bicgstab_solver))


Everything parametrised


TypeError: Error interpreting argument to <function _reduce_sum at 0x718ff5c9e700> as an abstract array. The problematic value is of type <class 'function'> and was passed to the function at path a.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.

In [70]:
result_everything = grad_theta_everything(theta_everything_base, MGOP, rhs_base, ltri_solver)

In [71]:
result_everything.vertices

Array([[-3.53301818e+01,  1.67505356e+02],
       [-4.13653885e+01,  4.96417518e+01],
       [-1.36833761e+02, -3.13685100e+01],
       [ 7.27304179e+01,  9.63942583e+00],
       [ 3.00407761e+00, -7.86182425e-01],
       [-2.45994267e+00, -2.58744312e-01],
       [ 2.82183466e+00, -8.02938972e-01],
       [ 1.50421640e+02, -2.41834236e+01],
       [-3.02861715e+01,  5.91805145e-01],
       [-2.60184545e+01,  1.08110014e+01],
       [ 1.11645223e+02, -5.55524211e+02],
       [-1.44620004e+02,  4.86333240e+02],
       [-1.17907429e+02,  3.85792949e+01],
       [ 2.29767560e+01,  2.09790810e+00],
       [ 2.25702390e+01,  1.30110995e+00],
       [-3.81177038e-01, -5.30763467e-01],
       [ 5.96801136e+00, -1.66392487e+00],
       [-4.99469533e+00, -5.11214920e-01],
       [ 1.42675394e+02, -2.27355326e+01],
       [ 1.75821768e+02, -3.17832970e+01],
       [-2.10256600e+02,  6.41933608e+01],
       [ 3.88167292e+02, -1.12969261e+03],
       [-1.86261605e+02,  5.34913763e+02],
       [-2.

In [20]:
@eqx.filter_jit
def solve_direct(MGOP, rhs, solver):    
    return jnp.sum(lx.linear_solve(MGOP, rhs, solver=solver).value)

In [44]:
filter_grad_total = eqx.filter_jit(eqx.filter_grad(solve_direct))
_filter_jit_f_to_memory(filter_grad_total, MGOP, rhs_base, ltri_solver)



{'peak': 198.24740600585938,
 'argument': 4.0999908447265625,
 'output': 0.20157623291015625,
 'temp': 194.28912353515625}

In [45]:
result_all_stupid = filter_grad_total(MGOP, rhs_base, ltri_solver)

In [49]:
result_all_stupid.transport_operator.solution_domain_data

SolutionDomainData(domain=Domain(vertices=Array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
      

In [17]:
filter_grad_total(theta_everything_base, MGOP, rhs_base, ltri_solver)

EverythingTheta(
  precomputed_geometric_data=PrecomputedGeometricData(
    vol_jacobians=f64[500,2,2],
    vol_jacobians_inv=f64[500,2,2],
    vol_jacobians_det=f64[500],
    face_jacobians_det=f64[500,3]
  ),
  cross_section_data=CrossSectionData(total=f64[8,4], scattering=f64[8,6,4,4])
)

In [19]:
grad_theta_everything(theta_everything_base, MGOP, rhs_base, ltri_solver)

EverythingTheta(
  precomputed_geometric_data=PrecomputedGeometricData(
    vol_jacobians=f64[500,2,2],
    vol_jacobians_inv=f64[500,2,2],
    vol_jacobians_det=f64[500],
    face_jacobians_det=f64[500,3]
  ),
  cross_section_data=CrossSectionData(total=f64[8,4], scattering=f64[8,6,4,4])
)

In [38]:
grad_theta = eqx.filter_jit(jax.grad(solve_result, argnums=0))

_filter_jit_f_to_memory(grad_theta, theta_base, MGOP_no_batch, rhs_base, ltri_solver)


{'peak': 32.47725772857666,
 'argument': 4.0900726318359375,
 'output': 0.006134033203125,
 'temp': 29.253173828125}

In [42]:
df_dtheta = grad_theta(theta_base, MGOP, rhs_base, ltri_solver)

In [45]:
solve_result(theta_base, MGOP, rhs_base, ltri_solver)

Array(740659.56725042, dtype=float64)

In [43]:
pytree_nbytes(df_dtheta)

np.int64(6400)

In [23]:
filter_grad_f = eqx.filter_jit(eqx.filter_grad(solve_result))#(MGOP, rhs_base, bicgstab_solver)

_filter_jit_f_to_memory(filter_grad_f, MGOP_no_batch, rhs_base, ltri_solver)

{'peak': 177.82476902008057,
 'argument': 4.0999908447265625,
 'output': 0.20157623291015625,
 'temp': 174.39093017578125}

In [24]:
_filter_jit_f_to_memory(filter_grad_f, MGOP_no_batch, rhs_base, bicgstab_solver)

{'peak': 177.82503604888916,
 'argument': 4.0999908447265625,
 'output': 0.20157623291015625,
 'temp': 174.10162353515625}

In [17]:
solve_result(MGOP, rhs_base, bicgstab_solver)

Array(740659.56725042, dtype=float64)

In [ ]:
solve_result(MGOP_no_batch, rhs_base, bicgstab_solver)

In [42]:
eqx.filter_grad(solve_fwd)(MGOP_no_batch, rhs_base, bicgstab_solver)

KeyboardInterrupt: 